In [1]:
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

model=load_model('model.h5')
## load the encoder and scaler 
with open('onehot_encoder_geo.pkl','rb') as file:
    onehot_encoder=pickle.load(file)

with open('label_encoder_gender.pkl','rb') as file:
    label_encoder=pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler=pickle.load(file)

## make a prediction
    


2026-11-03 21:17:07.426488: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3
2026-11-03 21:17:07.426522: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-11-03 21:17:07.426529: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-11-03 21:17:07.426557: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-11-03 21:17:07.426573: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [2]:
input_data={
    'CreditScore':600,
    'Geography':'France',
    'Gender':'Male',
    'Age':40,
    'Tenure':3,
    'Balance':60000,
    'NumOfProducts':2,
    'HasCrCard':1,
    'IsActiveMember':1,
    'EstimatedSalary':50000
}

In [3]:
#One-hot encode "Geography"
geo_encoded=onehot_encoder.transform([[input_data['Geography']]]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded,columns=onehot_encoder.get_feature_names_out(['Geography']))
geo_encoded_df

/Users/tirumaleshae/pythonproject/python-practice/annpractice1/venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [4]:
input_df=pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [5]:
input_df['Gender']=label_encoder.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [6]:
input_df=pd.concat([input_df.drop('Geography',axis=1),geo_encoded_df],axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [7]:
input_scaled=scaler.transform(input_df)
input_scaled



array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [8]:
prediction=model.predict(input_scaled)
prediction

2026-11-03 21:20:01.814488: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


1/1 [==============================] - 1s 945ms/step


array([[0.05345484]], dtype=float32)

In [10]:
prediction_proba=prediction[0][0]
prediction_proba

0.05345484

In [11]:
if prediction_proba>0.5:
    print('Customer will churn')
else:
    print('Customer will not churn')

Customer will not churn
